In [13]:
here::i_am("snakemake/01_create_arrow.R")
source(here::here("settings.R"))

# I/O
io$output.directory <- file.path(io$basedir,"ArchR_test2")
dir.create(file.path(io$output.directory), showWarnings = FALSE)

setwd(io$output.directory)

here() starts at /rds/project/rds-SDzz0CATGms/users/bt392/04_Rabbit_ATAC_final

Setting default number of Parallel threads to 1.



In [14]:
args = list()
args$sample = 'BGRGP2'
args$min_fragments = 10000
args$min_tss_score = 2

In [30]:
#genomeAnnotation = readRDS(file.path(io$basedir, 'genomeAnnotation.rds'))
geneAnnotation = readRDS(file.path(io$basedir, 'geneAnnotation_new.rds'))

library(BSgenome.Ocuniculus.NCBI.oryCun2)
genomeAnnotation = createGenomeAnnotation(
  genome = BSgenome.Ocuniculus.NCBI.oryCun2,
  chromSizes = NULL,
  blacklist = NULL,
  filter = FALSE,
  filterChr = c("chrM")
)
#genomeAnnotation$chromSizes = genomeAnnotation$chromSizes[1:30]
geneAnnotation$genes = geneAnnotation$genes[as.vector(seqnames(geneAnnotation$genes)) %in% genomeAnnotation$chromSizes@seqnames@values]
geneAnnotation$exons = geneAnnotation$exons[as.vector(seqnames(geneAnnotation$exons)) %in% genomeAnnotation$chromSizes@seqnames@values]
geneAnnotation$TSS = geneAnnotation$TSS[as.vector(seqnames(geneAnnotation$TSS)) %in% genomeAnnotation$chromSizes@seqnames@values]

Getting genome..

Getting chromSizes..

Getting blacklist..

Blacklist not downloaded! Continuing without, be careful for downstream biases..



In [36]:
length(geneAnnotation$genes[geneAnnotation$TSS@seqnames@values != 'chrM',]@seqnames@values)

[1] 1401

In [17]:
# trim 2kb ends of geneAnnotation, otherwise gives error: 

exclude = GRanges(
    seqnames = Rle(rep(genomeAnnotation$chromSizes@seqnames@values,2)),
    ranges = IRanges(start = c(genomeAnnotation$chromSizes@ranges@start, 
                               genomeAnnotation$chromSizes@ranges@width-2000), 
                     end = c(genomeAnnotation$chromSizes@ranges@start + 2000, 
                             genomeAnnotation$chromSizes@ranges@width)))

exclude_ranges = as.data.frame(findOverlaps(exclude, geneAnnotation$TSS))
if(nrow(exclude_ranges)){
geneAnnotation$TSS = geneAnnotation$TSS[-exclude_ranges$subjectHits]
}

exclude_ranges = as.data.frame(findOverlaps(exclude, geneAnnotation$genes))
if(nrow(exclude_ranges)){
geneAnnotation$genes = geneAnnotation$genes[-exclude_ranges$subjectHits]
}
exclude_ranges = as.data.frame(findOverlaps(exclude, geneAnnotation$exons))
if(nrow(exclude_ranges)){
geneAnnotation$exons = geneAnnotation$exons[-exclude_ranges$subjectHits]
}

In [18]:
geneAnnotation$TSS
geneAnnotation$genes
geneAnnotation$exons
genomeAnnotation$chromSizes 

GRanges object with 39214 ranges and 2 metadata columns:
           seqnames    ranges strand |     tx_id            tx_name
              <Rle> <IRanges>  <Rle> | <integer>        <character>
      [1]      chr1     15188      + |         1 ENSOCUT00000014253
      [2]      chr1     20324      + |         2 ENSOCUT00000038745
      [3]      chr1     52453      + |         3 ENSOCUT00000005053
      [4]      chr1     75200      + |         4 ENSOCUT00000005041
      [5]      chr1    125890      + |         5 ENSOCUT00000005032
      ...       ...       ...    ... .       ...                ...
  [39210] chrUn0008   5509534      - |     40010 ENSOCUT00000002990
  [39211] chrUn0008   5509534      - |     40011 ENSOCUT00000013429
  [39212] chrUn0008   5509534      - |     40012 ENSOCUT00000050100
  [39213] chrUn0008   5544740      - |     40013 ENSOCUT00000025013
  [39214] chrUn0008   5569951      - |     40014 ENSOCUT00000013411
  -------
  seqinfo: 1402 sequences from an unspecified gen

GRanges object with 21773 ranges and 2 metadata columns:
           seqnames          ranges strand |            gene_id
              <Rle>       <IRanges>  <Rle> |        <character>
      [1]      chr1     15188-30379      + | ENSOCUG00000014251
      [2]      chr1     52453-53038      + | ENSOCUG00000005054
      [3]      chr1     57421-74906      - | ENSOCUG00000005046
      [4]      chr1     75200-85749      + | ENSOCUG00000005044
      [5]      chr1     92423-95846      - | ENSOCUG00000005040
      ...       ...             ...    ... .                ...
  [21769] chrUn0008 5475029-5509535      - | ENSOCUG00000013420
  [21770] chrUn0008 5526270-5544741      - | ENSOCUG00000024159
  [21771] chrUn0008 5554610-5569952      - | ENSOCUG00000013413
  [21772] chrUn0008 5603833-5682641      + | ENSOCUG00000038015
  [21773] chrUn0008 5698008-5699199      + | ENSOCUG00000030550
                      symbol
                 <character>
      [1]              WDR31
      [2]             RN

GRanges object with 181338 ranges and 3 metadata columns:
            seqnames          ranges strand |   exon_id            gene_id
               <Rle>       <IRanges>  <Rle> | <integer>        <character>
       [1]      chr1     15188-15282      + |         1 ENSOCUG00000014251
       [2]      chr1     20324-20326      + |         2 ENSOCUG00000014251
       [3]      chr1     20453-20603      + |         3 ENSOCUG00000014251
       [4]      chr1     20453-20603      + |         4 ENSOCUG00000014251
       [5]      chr1     21033-21163      + |         5 ENSOCUG00000014251
       ...       ...             ...    ... .       ...                ...
  [181334] chrUn0008 5562851-5562913      - |    183454 ENSOCUG00000013413
  [181335] chrUn0008 5565201-5565248      - |    183455 ENSOCUG00000013413
  [181336] chrUn0008 5566282-5566364      - |    183456 ENSOCUG00000013413
  [181337] chrUn0008 5567581-5567703      - |    183457 ENSOCUG00000013413
  [181338] chrUn0008 5569667-5569952      

GRanges object with 30 ranges and 0 metadata columns:
        seqnames      ranges strand
           <Rle>   <IRanges>  <Rle>
   [1]      chr1 1-194850757      *
   [2]     chr10  1-47997241      *
   [3]     chr11  1-87554214      *
   [4]     chr12 1-155355395      *
   [5]     chr13 1-143360832      *
   ...       ...         ...    ...
  [26] chrUn0004   1-7969563      *
  [27] chrUn0005   1-6940555      *
  [28] chrUn0006   1-6663442      *
  [29] chrUn0007   1-6124611      *
  [30] chrUn0008   1-5787221      *
  -------
  seqinfo: 3242 sequences from an unspecified genome

In [19]:
fragment_file_path = paste0(io$basedir, '/data/', args$sample, '_fragments.tsv.gz')

In [ ]:

#Create Arrow File
ArrowFiles <- createArrowFiles(
  inputFiles = fragment_file_path,
  sampleNames = paste0('rabbit_', args$sample),
    
  minTSS = args$min_tss_score, #Dont set this too high because you can always increase later
  minFrags = args$min_fragments , 
    
  addTileMat = TRUE,
  addGeneScoreMat = TRUE,
    
  geneAnnotation = geneAnnotation,
  genomeAnnotation = genomeAnnotation, 
    force= TRUE
)

ArchR logging to : ArchRLogs/ArchR-createArrows-9b835d2e73e5-Date-2022-01-27_Time-22-22-24.log
If there is an issue, please report to github with logFile!

Cleaning Temporary Files

2022-01-27 22:22:25 : Batch Execution w/ safelapply!, 0 mins elapsed.

(rabbit_BGRGP1 : 1 of 1) Checking if completed file exists!

2022-01-27 22:22:25 : (rabbit_BGRGP1 : 1 of 1) Arrow Exists! Overriding since force = TRUE!, 0 mins elapsed.

(rabbit_BGRGP1 : 1 of 1) Determining Arrow Method to use!

2022-01-27 22:22:25 : (rabbit_BGRGP1 : 1 of 1) Reading In Fragments from inputFiles (readMethod = tabix), 0.001 mins elapsed.

2022-01-27 22:22:25 : (rabbit_BGRGP1 : 1 of 1) Tabix Bed To Temporary File, 0.001 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-27 22:25:21 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 2 Percent, 2.931 mins elapsed.

Warning message in sprintf("%s Reading Tabix

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-27 22:52:27 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 62 Percent, 30.028 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-27 22:52:37 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 64 Percent, 30.195 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-27 22:52:46 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 66 Percent, 30.352 mins elapsed.

Warning message in sprintf("%s Reading TabixFile %s Percent", prefix, round(100 * :
“one argument not used by format '%s Reading TabixFile %s Percent'”
2022-01-27 22:52:56 : (rabbit_BGRGP1 : 1 of 1) Reading TabixFile 68 Percent, 30.512 mins elapse

In [7]:
# Calculate doublet scores

ArrowFile = paste0(io$output.directory, '/rabbit_', args$sample, '.arrow')

doubScores <- addDoubletScores(
  input = ArrowFiles,
  k = 15, #Refers to how many cells near a "pseudo-doublet" to count.
  knnMethod = "UMAP", #Refers to the embedding to use for nearest neighbor search.
  LSIMethod = 1
)


ArchR logging to : ArchRLogs/ArchR-addDoubletScores-9b83368b7997-Date-2022-01-27_Time-20-49-43.log
If there is an issue, please report to github with logFile!

2022-01-27 20:49:44 : Batch Execution w/ safelapply!, 0 mins elapsed.

ArchR logging successful to : ArchRLogs/ArchR-addDoubletScores-9b83368b7997-Date-2022-01-27_Time-20-49-43.log

